# Money brain on a Colab A100

Runs the whole system (`docs/COLAB.md`).
- **GPU:** only the language model (Qwen3-32B, served by vLLM) reads and explains.
- **CPU:** the memory bank, the evidence engine, your chain engine and the scorekeeper, as on any machine.

Paper only: nothing here places orders. Public data only (`CLAUDE.md`).

**Before you start:**
- Runtime → Change runtime type → **A100 GPU**.
- Optional Colab secrets (the key icon on the left):
  - `SEC_USER_AGENT` (`Your Name your@email.com`): the SEC requires it.
  - `GITHUB_TOKEN`: only if the repo is private, or to push results.

## 1. GPU and model choice

In [ ]:
import subprocess, json
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
mem_gb = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                            capture_output=True, text=True).stdout.split()[0]) / 1024
# 80 GB: the full model. 40 GB: the official 4-bit (AWQ) build. Override MODEL here if you want another one.
MODEL = "Qwen/Qwen3-32B" if mem_gb > 70 else "Qwen/Qwen3-32B-AWQ"
MAX_LEN = 32768
print(f"{mem_gb:.0f} GB GPU -> {MODEL}")

## 2. Storage: the bank lives on Drive, work happens on local disk
Colab wipes the machine when the session ends.
- The bank is copied from Drive to `/content/var` at the start (fast local disk).
- It is copied back at the end, in section 12.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_VAR = "/content/drive/MyDrive/money/var"
LOCAL_VAR = "/content/var"
os.makedirs(DRIVE_VAR, exist_ok=True); os.makedirs(LOCAL_VAR, exist_ok=True)
!rsync -a --info=progress2 "{DRIVE_VAR}/" "{LOCAL_VAR}/"
# An archive made on another machine (tar czf bank.tgz var/bank var/series) and uploaded to MyDrive/money/:
if not os.path.exists(f"{LOCAL_VAR}/bank") and os.path.exists("/content/drive/MyDrive/money/bank.tgz"):
    !tar xzf /content/drive/MyDrive/money/bank.tgz -C /content && echo restored from bank.tgz
!du -sh {LOCAL_VAR} 2>/dev/null

## 3. The code

In [ ]:
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
tok = secret("GITHUB_TOKEN")
url = f"https://{tok}@github.com/nikku03/money" if tok else "https://github.com/nikku03/money"
if not os.path.exists("/content/money"):
    !git clone -q -b claude/upbeat-turing-z9jk2b {url} /content/money
%cd /content/money
!git log --oneline -1
!pip install -q -e .

## 4. The language model (vLLM), in the background

In [ ]:
!pip install -q vllm

In [ ]:
import subprocess, time, urllib.request
log = open("/content/vllm.log", "w")
vllm = subprocess.Popen(["vllm", "serve", MODEL, "--max-model-len", str(MAX_LEN), "--gpu-memory-utilization", "0.90",
                         "--enable-auto-tool-choice", "--tool-call-parser", "hermes", "--port", "8000"],
                        stdout=log, stderr=subprocess.STDOUT)
for i in range(180):                     # the first start downloads the weights: up to ~15 minutes
    try:
        served = json.load(urllib.request.urlopen("http://127.0.0.1:8000/v1/models"))["data"][0]["id"]
        print("model ready:", served); break
    except Exception:
        time.sleep(10)
else:
    print("not up yet; see /content/vllm.log")

## 5. Settings the code reads

In [ ]:
os.environ.update({
    "MONEY_HOME": LOCAL_VAR,
    "SEC_USER_AGENT": secret("SEC_USER_AGENT") or "money-brain research you@example.com",  # put your own in the secret
    "BRAIN_LLM_URL": "http://127.0.0.1:8000/v1",
    "BRAIN_LLM_MODEL": MODEL,
    "BRAIN_LLM_CTX": str(MAX_LEN),
    "BRAIN_LLM_TIMEOUT": "600",
})
from brain import llm
print(llm.available())

## 6. Check the code and the bank (optional, about 3 minutes)

In [ ]:
!MONEY_HOME=/tmp/testhome python -m unittest discover -s tests -t . 2>&1 | tail -3
!python -m brain check | tail -2

## 7. Data
- **If section 2 restored a bank:** run only the daily refresh.
- **On a fresh start:** set `REBUILD = True`. The first build takes a few hours: SEC filings and prices for 741 stocks, plus insider history.

In [ ]:
REBUILD = False
if REBUILD:
    !python -m brain ingest-universe --broad
    !python -m brain insiders
    !python -m brain markets
    !python -m brain macro
WATCHLIST = ["AAPL", "NVDA", "MSFT"]
WL = " ".join(WATCHLIST)
!python -m brain ingest {WL} SPY        # the daily refresh: prices, filings, headlines
!python -m brain alerts check

## 8. Your chain engine: cause → effect

In [ ]:
!python -m brain chain train
!python -m brain chain follow oil_price_up
!python -m brain chain follow inflation_up --hops 5

## 9. Read, decide, forecast (the model reads; the engine decides)
- **`scout`:** the model reads public pages and notes claims with their sources. It has no web search unless `BRAIN_SEARX_URL` is set.
- **`decide`:** the model makes its own call. Policy D1 weighs only evidence with a proven record, so expect NO_EDGE today.
- **`forecast`:** price ranges and the chance to beat SPY. The first run trains the live model, which takes 5-10 minutes.

In [ ]:
!python -m brain scout AAPL --max-pages 4
!python -m brain decide {WL}
!python -m brain forecast AAPL --llm

## 10. The terminal, through Colab's port proxy
The terminal only answers its own machine. Setting `BRAIN_TERMINAL_PROXY_HOST` lets it accept the one address Colab's proxy uses; the per-session token is still required. Open the window, then type `AAPL DEC`, `AAPL FCST`, or ASK questions.

In [ ]:
from google.colab import output
from urllib.parse import urlparse
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8765)")
os.environ["BRAIN_TERMINAL_PROXY_HOST"] = urlparse(proxy_url).hostname
term = subprocess.Popen(["python", "-m", "brain", "terminal", "--no-browser", "--port", "8765"],
                        stdout=open("/content/terminal.log", "w"), stderr=subprocess.STDOUT)
time.sleep(20)
output.serve_kernel_port_as_window(8765)

## 11. Score: how did past decisions, forecasts and sources do?

In [ ]:
!python -m brain decide score
!python -m brain forecast score

## 12. Save
- The bank goes back to Drive.
- The research ledger and RESULTS.md live in the repo: push them with a token, or download them.

In [ ]:
!rsync -a --info=progress2 "{LOCAL_VAR}/" "{DRIVE_VAR}/"
!git status --short research/
# With GITHUB_TOKEN set and something to save:
# !git add research && git commit -m "Colab run" && git push origin claude/upbeat-turing-z9jk2b

## 13. Testing the bigger model honestly
A 32B model reads and explains far better than the 1.7B one tested so far. Whether it **predicts** better is a separate question, and only its record can answer it.
- **Its record starts now.** Every `forecast --llm` run in section 9 is recorded as an F1-llm fact. Each one is scored once its sessions have passed (`forecast score`, section 11).
- **Run it weekly on the same stocks.** That builds a record on weeks the model cannot have seen.
- **The past window is used up.** F1d already holds a result for the last 8 weeks, and the code refuses a new test on a window that overlaps one with a result. A declared retest needs a new test id with a new window, written into `brain/research/forecast_tests.py` and committed before it runs.

In [ ]:
vllm.terminate()      # stop the model server when you are done (the GPU is billed while it runs)